# Bahth بحث — image search in Arabic, Turkish and English

## Part 2: fine-tuning the text encoder

Part 1 showed the gap: a compact multilingual CLIP finds the right photo first for 49% of English queries but only about 25% of Arabic and Turkish ones. Here its text encoder is fine-tuned three ways, from the same starting point, and each result is scored twice: on Flickr8k test photos (the kind it trained on) and on XTD (MS-COCO photos it never trained on).

Code: https://github.com/readaloush/bahth

**Settings:** Internet on, GPU accelerator.

In [ ]:
!pip -q install sentence-transformers
!git clone -q https://github.com/readaloush/bahth.git
%cd bahth
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
!python -m bahth.train --data data --steps 1200 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$'

In [ ]:
import json, pandas as pd
r = json.load(open('results/finetune.json'))
rows = [{'recipe': n, 'test set': s, 'language': l, **{k: round(v * 100, 1) for k, v in m.items() if k.startswith('R@')}} for n, d in r['runs'].items() for s, mm in d['metrics'].items() for l, m in mm.items()]
print('selected on dev:', r.get('selected_on_dev'), '| student parameters:', r['parameters'])
pd.DataFrame(rows).pivot_table(index=['test set', 'language'], columns='recipe', values='R@1')

In [ ]:
!ls -la results results/student 2>/dev/null | head -30